## 0. Baseline model

In [ ]:
from convtasnet import ConvTasNet, SI_SNR_PIT
from hw_utils import prep_dataset, LibriMixDataset

import lighter

import os

import torch
from torch.utils.data import DataLoader


device = 'cuda' if torch.cuda.is_available() else 'cpu'

config = {
    'working-dir' : './_fit/hw1/',
    'dataset' : {
        'root'              : './data/mini_libri2mix/', 
        'segment_length'    : 3, 
        'typ'               : 'clean', 
        'sample_rate'       : 8000,
    },
    'dataloader' : {
        'batch_size'        : 8,
        'shuffle'           : True,
        'num_workers'       : 4,
        'prefetch_factor'   : 4,
    },
    'model' : {
        'N' : 64,
        'L' : 20,
        'B' : 64,
        'H' : 64,
        'P' : 3,
        'X' : 2,
        'R' : 4,
        'C' : 2,
        'mask_nonlinear' : 'sigmoid',
    },
    'optimizer' : {
        'lr'            : 1e-3,
        'weight_decay'  : 0,
    },
    'fit' : {
        'epochs' : 5,
    }
}

prep_dataset('./data/mini_libri2mix/')

train_dataset = LibriMixDataset(
    subset="train", 
    **config['dataset']
)

val_dataset = LibriMixDataset(
    subset="val", 
    **config['dataset']
)

train_loader = DataLoader(train_dataset, **config['dataloader'])
val_loader   = DataLoader(  val_dataset, **config['dataloader'])

model = ConvTasNet(**config['model'])

model.compile(
    torch.optim.Adam(model.parameters(), **config['optimizer']),
    SI_SNR_PIT(),
    metrics=[],
    device=device,
)

chkpoint_path = os.path.join(
    config['working-dir'], 'ConvTasNet/model_{epoch}.pth'
)
log_path = os.path.join(
    config['working-dir'], 'ConvTasNet_logs.csv'
)

train_l, val_l = model.fit(
    train_loader,
    validation_loader=val_loader,
    callbacks=[
        lighter.callbacks.CSVLogger(log_path),
        lighter.callbacks.History(),
        lighter.callbacks.Checkpoint(
            chkpoint_path,
            save_best_only=True
        ),
    ],
    **config['fit'],
)


In [ ]:
from lighter.utils import plot_loss
import matplotlib.pyplot as plt

plot_loss(train_l, val_l)
plt.show()


## 1. Noise Levels

## 2. Evaluation of the original model

In [ ]:



model.evaluate(
    val_loader,
    callbacks = [
        lighter.callbacks.History(),
    ],
)


## 3. Training of a noise-robust model